# register-buffer — faded example 1: Register a Causal Mask as a Buffer (Faded)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-buffer`. Running the beacon reports progress on the `PyTorch: register_buffer` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: register_buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-buffer`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-buffer"
DD_SUBTOPIC = "PyTorch: register_buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Position encodings and attention masks are computed once at construction time and should travel with the model's state. Use `self.register_buffer(name, tensor)` to attach them so they appear in `state_dict()` and move to the right device with `.to(device)`, without being treated as learnable parameters.

## Faded exercise 1

A `CausalAttention` module needs a causal mask stored as a buffer named `'causal_mask'`. The mask is a lower-triangular boolean tensor of shape `(seq_len, seq_len)`.

The module skeleton is provided with the `nn.Parameter` and the forward method. Your task is to **register the causal mask as a buffer** in `__init__`.

**Fill in:** Register the lower-triangular boolean tensor as a buffer named 'causal_mask' using self.register_buffer.

In [ ]:
import torch as t
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, seq_len: int, d_model: int):
        super().__init__()
        self.proj = nn.Parameter(t.randn(d_model, d_model) * 0.02)
        mask = t.tril(t.ones(seq_len, seq_len, dtype=t.bool))
        self.register_buffer('causal_mask', mask)

    def forward(self, scores: t.Tensor) -> t.Tensor:
        return scores.masked_fill(~self.causal_mask, float('-inf'))


def _test():
    import torch as t
    t.manual_seed(7)
    m = CausalAttention(seq_len=4, d_model=8)
    # causal_mask should be in buffers, not parameters
    buffer_names = [n for n, _ in m.named_buffers()]
    param_names  = [n for n, _ in m.named_parameters()]
    assert 'causal_mask' in buffer_names, 'causal_mask missing from buffers'
    assert 'causal_mask' not in param_names, 'causal_mask should not be a parameter'
    assert 'causal_mask' in m.state_dict(), 'causal_mask missing from state_dict'
    assert not m.causal_mask.requires_grad, 'buffer should have requires_grad=False'
    expected = t.tril(t.ones(4, 4, dtype=t.bool))
    assert t.equal(m.causal_mask, expected), 'mask values incorrect'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, seq_len: int, d_model: int):
        super().__init__()
        self.proj = nn.Parameter(t.randn(d_model, d_model) * 0.02)
        mask = t.tril(t.ones(seq_len, seq_len, dtype=t.bool))
        self.register_buffer('causal_mask', mask)

    def forward(self, scores: t.Tensor) -> t.Tensor:
        return scores.masked_fill(~self.causal_mask, float('-inf'))
```
</details>